# 07 - Monte Carlo, uncertainty and rare events

## Learning objectives

- Interpret Monte Carlo estimates and standard errors
- Compare simulated and analytic FTA results
- Propagate parameter uncertainty
- Recognize rare-event difficulties and splitting/RESTART principles


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Repository root:", ROOT)


## 1. Monte Carlo principle

Generate many random realizations, evaluate the system in each realization, and estimate from outcome frequencies. For a Bernoulli event, $\hat p=N_{event}/N$. For sufficiently large event counts, $SE(\hat p)\approx\sqrt{\hat p(1-\hat p)/N}$.


In [ ]:
rng=np.random.default_rng(42); N=10_000; p=.02
x=rng.random(N)<p; phat=x.mean(); se=np.sqrt(phat*(1-phat)/N)
print(phat,se,x.sum())


## 2. Convergence

Increasing sample size need not improve every individual run monotonically. The typical uncertainty scale decreases as $1/\sqrt N$.


In [ ]:
rows=[]
for n in [100,500,1_000,5_000,10_000,50_000,100_000]:
 r=np.random.default_rng(100+n); rows.append({'N':n,'estimate':np.less(r.random(n), p).mean()})
conv=pd.DataFrame(rows); plt.semilogx(conv.N,conv.estimate,'o-'); plt.axhline(p,ls='--'); plt.grid(alpha=.3); plt.show(); conv



## 3. TK-101 FTA by Monte Carlo

Generate independent Bernoulli basic events. The simulator deliberately uses the same independence assumptions as the analytic tree. Agreement is a numerical implementation check, not evidence that those assumptions match a real plant.


In [ ]:
prob={'CHALLENGE':.08,'LSHH_FOD':.01,'XV_FOD':.005,'LT_LOW':.02,'OP_MISS':.04}
def simulate(n=500_000,seed=42):
 r=np.random.default_rng(seed); s = {k: np.less(r.random(n), v) for k, v in prob.items()}
 auto=s['LSHH_FOD']|s['XV_FOD']; manual=s['LT_LOW']|s['OP_MISS']; over=s['CHALLENGE']&auto&manual
 ph=over.mean(); return ph,np.sqrt(ph*(1-ph)/n),int(over.sum())
ph,se,events=simulate(); print(ph,se,events)
p_auto=1-(1-.01)*(1-.005); p_manual=1-(1-.02)*(1-.04); exact=.08*p_auto*p_manual
print('exact=',exact,'z=',(ph-exact)/se)


## 4. Parameter uncertainty

Input probabilities are uncertain too. Use Beta distributions for illustration. The distribution parameters are **synthetic**, not operational evidence.


In [ ]:
r=np.random.default_rng(7); M=20_000
p_ch=r.beta(8,92,M); p_lshh=r.beta(2,198,M); p_xv=r.beta(1,199,M); p_lt=r.beta(4,196,M); p_op=r.beta(8,192,M)
p_auto=1-(1-p_lshh)*(1-p_xv); p_manual=1-(1-p_lt)*(1-p_op); p_over=p_ch*p_auto*p_manual
print(np.quantile(p_over,[.05,.5,.95])); plt.hist(p_over,bins=60); plt.xlabel('P(overflow)'); plt.show()


## 5. The rare-event problem

For $p=10^{-7}$ and $N=10^6$, only 0.1 events are expected. Zero observations do not imply zero probability.


In [ ]:
rare_p=1e-7
for n in [10**4,10**5,10**6,10**7]: print(f'N={n:,}, expected event count={n*rare_p:.3f}')


## 6. Splitting and the RESTART principle

Intermediate levels identify trajectories approaching a rare outcome. Replicate or reuse those trajectories to concentrate computation on relevant paths. The supplied function is a **pedagogical multilevel-splitting demonstration**, not a production-grade RESTART implementation.


In [ ]:
from safetycourse.simulation import random_walk_exceedance,multilevel_splitting_random_walk
crude,se=random_walk_exceedance(n=20_000,steps=50,drift=-.05,sigma=1.,threshold=20,seed=1)
split=multilevel_splitting_random_walk(particles=2_000,steps=50,drift=-.05,sigma=1.,levels=[8,14,20],seed=1)
print('crude',crude,'SE',se); print('splitting demo',split)


## Try it!

1. Divide every basic-event probability by ten and observe the event count in 500,000 runs.
2. Increase N and inspect the standard error.
3. Change Beta concentration while preserving the means.
4. Change the splitting levels and compare stability.


## Summary

Monte Carlo is general but can be expensive for rare events. Next, investigate how **dependencies and common-cause failures** undermine idealized redundancy benefits.
